# Лабораторная работа 9. Рекуррентные сети и трансформеры: архитектуры для последовательностей и текста

**Курс:** Машинное обучение. Безопасность ИИ-систем

**По материалам лекции 7**


**Вариант / seed:** _номер варианта или значение SEED_


### Таблица вариантов

| Вариант | `SEED` | `D_MODEL` | `N_MAX` (пар в обучении) | `D_CELL` | Температуры для части H |
|---|---|---|---|---|---|
| 1 | 42 | 64 | 12 | 48 | 0.5 / 1.0 / 1.5 |
| 2 | 7 | 48 | 10 | 40 | 0.4 / 1.0 / 1.6 |
| 3 | 123 | 64 | 8 | 48 | 0.6 / 1.0 / 1.4 |
| 4 | 2024 | 56 | 12 | 56 | 0.5 / 1.2 / 1.8 |
| 5 | 314 | 48 | 8 | 40 | 0.7 / 1.0 / 1.3 |
| 6 | 555 | 64 | 10 | 56 | 0.5 / 0.9 / 1.7 |

> Вариант выдаёт преподаватель. Все числа в отчёте должны соответствовать **вашему** варианту,
> а не значениям из примера в тексте.

### Цель работы

Экспериментально установить, чем механизм внимания отличается от рекуррентного накопления состояния,
какой ценой достигается каждое из свойств (точность вспоминания, стоимость контекста, память при
генерации), и какие архитектурные свойства становятся поверхностью атаки на языковые модели.

### Задачи

1. Реализовать BPE-токенизацию и оценить влияние уровня токенизации на длину последовательности.
2. Измерить затухание градиента в рекуррентной сети и сравнить мультипликативный путь с аддитивным.
3. Реализовать масштабированное скалярное внимание с каузальной маской.
4. Проверить численно, что RoPE зависит только от разности позиций, и построить штраф ALiBi.
5. Обучить двухслойный мини-трансформер задаче ассоциативного вспоминания и найти индукционную голову.
6. Реализовать линейное внимание в рекуррентной форме и сравнить его с softmax-вниманием.
7. Сделать рекуррентную ячейку избирательной и проверить, что это даёт управляемую память.
8. Реализовать top-k и top-p отсечение и оценить статистику многих попыток генерации.

### Порядок выполнения

1. Выполните ячейку настройки среды и впишите параметры своего варианта.
2. Идите по частям A–H подряд, заполняя каждый блок `TODO`.
3. После каждой части ответьте на вопрос в конце этой части (ячейка Markdown «Ответ»).
4. Заполните таблицы отчёта и сформулируйте выводы.
5. Перед сдачей выполните Kernel → Restart & Run All и убедитесь, что ошибок нет.

### Ограничение по безопасности

Работа не содержит и не должна содержать кода атак на реальные языковые модели и сервисы,
готовых строк обхода ограничений или инструментов автоматического подбора вредоносных запросов.
Часть H исследует **статистику выборки и архитектурные предпосылки** уязвимостей на игрушечной
модели — этого достаточно для понимания следующей лекции.

In [ ]:
# Настройка окружения и единый стиль рисунков
import math, time, warnings
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
torch.set_num_threads(2)

PALETTE = {"attn": "#2E86AB", "linear": "#F2A541", "bad": "#D6455F",
           "good": "#3FA46A", "accent": "#7B4FA0", "gray": "#8A94A6"}
CMAP_HEAT = LinearSegmentedColormap.from_list(
    "heat7", ["#0B1D33", "#2E86AB", "#F2E8CF", "#F2A541", "#D6455F"])

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10.5,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10.5,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "--",
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

def finish(fig, title=None):
    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.96 if title else 1.0))
    plt.show()

print("torch:", torch.__version__, "| потоков:", torch.get_num_threads())

In [ ]:
# TODO 0: подставьте параметры СВОЕГО варианта из таблицы вариантов
SEED    = 42      # <-- ваш SEED
D_MODEL = 64      # <-- ваш D_MODEL
N_MAX   = 12      # <-- ваше число пар в обучении
D_CELL  = 48      # <-- ваша размерность рекуррентной ячейки
TAUS    = (0.5, 1.0, 1.5)   # <-- ваши температуры для части H

np.random.seed(SEED)
torch.manual_seed(SEED)
print("вариант: SEED=%d, D_MODEL=%d, N_MAX=%d, D_CELL=%d, TAUS=%s"
      % (SEED, D_MODEL, N_MAX, D_CELL, str(TAUS)))

## Часть A. Токенизация: BPE с нуля

Модель работает не с буквами и не со словами, а с токенами. Уровень токенизации задаёт длину
последовательности $n$, а значит и стоимость внимания $O(n^2)$, а также распределение частот:
редкие токены остаются недообученными.

**Задание A.1.** Реализуйте один шаг BPE: подсчёт частот пар соседних символов и слияние самой
частой пары.
**Задание A.2.** Обучите BPE на корпусе и измерьте коэффициенты сжатия «символы → BPE» и
«BPE → слова».
**Задание A.3.** Постройте график длины корпуса в токенах от числа выполненных слияний и
распределение частот токенов в логарифмических осях; отметьте порог редких токенов
`RARE_THR` (частота не выше `RARE_THR` за весь корпус).

In [ ]:
CORPUS = (
    "внимание позволяет модели обращаться к любой позиции последовательности напрямую. "
    "рекуррентная сеть накапливает состояние и постепенно теряет информацию о далёком прошлом. "
    "трансформер обучается параллельно по всем позициям и потому масштабируется. "
    "позиционное кодирование сообщает модели порядок токенов. "
    "линейное внимание заменяет матрицу внимания рекуррентным состоянием. "
    "избирательная рекуррентность делает затухание зависящим от входа. "
    "квантование и кэширование уменьшают стоимость генерации. "
) * 6

words = CORPUS.split()
vocab_words = {}
for w in words:
    key = tuple(w) + ("</w>",)
    vocab_words[key] = vocab_words.get(key, 0) + 1

def pair_counts(vocab):
    # TODO A.1a: верните словарь {пара_соседних_символов: суммарная частота}
    #            вес пары = частота слова, в котором она встретилась
    raise NotImplementedError("Задание A.1a")

def merge_pair(vocab, pair):
    # TODO A.1b: верните новый словарь, в котором все вхождения pair склеены в один символ
    raise NotImplementedError("Задание A.1b")

print("слов в корпусе:", len(words), "| уникальных словоформ:", len(vocab_words))

In [ ]:
# TODO A.2: выполните N_MERGES слияний, на каждом шаге сохраняя длину корпуса в токенах
N_MERGES = 120
vocab = dict(vocab_words)
lengths, merges = [], []

# TODO A.2a: цикл слияний: pair_counts -> самая частая пара -> merge_pair
#            длина корпуса = sum(len(seq) * freq for seq, freq in vocab.items())

# TODO A.2b: посчитайте и напечатайте коэффициенты сжатия
n_chars = len(CORPUS.replace(" ", ""))
raise NotImplementedError("Задание A.2")

In [ ]:
# TODO A.3: рисунок A: длина корпуса от числа слияний + частоты токенов (закон Ципфа)
# слева: lengths от номера слияния
# справа: отсортированные по убыванию частоты токенов в log-log осях,
#         отметьте горизонтальной линией порог «редкий токен» (частота <= RARE_THR)
RARE_THR = 6
raise NotImplementedError("Задание A.3")

**Вопрос A.** Во сколько раз выросла бы стоимость слоя внимания, если перейти от BPE к посимвольной
токенизации при том же тексте? Используйте измеренный коэффициент сжатия и зависимость $O(n^2)$.

**Ответ A.**

## Часть B. Затухание градиента в рекуррентной сети

Для рекуррентности $h_t = \phi(W h_{t-1} + U x_t)$ производная по состоянию, удалённому на $k$ шагов,
содержит произведение $k$ матриц:

$
\frac{\partial h_T}{\partial h_{T-k}} = \prod_{j=0}^{k-1} \operatorname{diag}(\phi')\, W .
$

Норма такого произведения ведёт себя как $\rho(W)^k$. Аддитивный путь с гейтом $f_t$ даёт вместо
этого множитель $\prod f_t$, который при $f_t \to 1$ почти не затухает.

**Задание B.1.** Вычислите норму произведения $k$ матриц с заданным спектральным радиусом.
**Задание B.2.** Постройте график нормы от $k$ для $\rho \in \{0.9, 1.0, 1.2\}$ и добавьте
аддитивные пути с $f_t = 0.99$ и $f_t = 0.95$.
**Задание B.3.** Найдите эффективный горизонт памяти: при каком $k$ норма падает ниже $10^{-6}$.

In [ ]:
def make_W(d, rho, gen):
    # матрица со заданным спектральным радиусом
    A = gen.normal(size=(d, d)) / np.sqrt(d)
    return A * rho / np.max(np.abs(np.linalg.eigvals(A)))

def grad_norms(rho, k_max=120, d=32, seed=0):
    # TODO B.1: верните массив норм произведения k матриц W для k = 1..k_max
    #           используйте np.linalg.norm(P, 2) и накопление P = P @ W
    raise NotImplementedError("Задание B.1")

def horizon(rho, thr=1e-6, k_max=4000, d=32, seed=0):
    # TODO B.3: верните наименьшее k, при котором норма падает ниже thr
    #           (оценивайте норму как rho ** k, чтобы не перемножать тысячи матриц)
    raise NotImplementedError("Задание B.3")

print("подсказка: rho ** k = thr  ->  k = log(thr) / log(rho)")

In [ ]:
# TODO B.2: рисунок B: слева нормы от k в логарифмической шкале для трёх значений rho
#           и две линии аддитивного пути f**k (f = 0.99 и 0.95);
#           справа эффективный горизонт памяти horizon(rho) для rho от 0.8 до 0.999
raise NotImplementedError("Задание B.2")

**Вопрос B.** Пусть требуется удерживать зависимость длиной 500 шагов с ослаблением сигнала не
более чем в 10 раз. Какое значение гейта $f_t$ для этого нужно? Сравните с $\rho(W)$, необходимым в
чистой рекуррентности.

**Ответ B.**

## Часть C. Внимание с нуля и роль масштабирования

$
\operatorname{Attention}(Q,K,V) = \operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V .
$

Делитель $\sqrt{d_k}$ не косметика: при $q,k \sim \mathcal N(0,1)$ скалярное произведение имеет
дисперсию $d_k$, и без нормировки softmax при больших $d_k$ вырождается почти в one-hot, что
обнуляет градиент по ключам.

**Задание C.1.** Реализуйте одну голову внимания с каузальной маской и без неё.
**Задание C.2.** Проверьте два обязательных свойства: строки матрицы внимания суммируются в единицу,
а при каузальной маске верхний треугольник строго нулевой.
**Задание C.3.** Измерьте среднюю энтропию распределения внимания как функцию $d_k$ с делителем и
без него, постройте рисунок с картами внимания и графиком энтропии.

In [ ]:
def attention(Q, K, V, causal=True, scale=True):
    # TODO C.1: реализуйте scaled dot-product attention
    #   1) scores = Q @ K.T
    #   2) если scale: разделите на sqrt(d_k)
    #   3) если causal: запретите будущее (значение -inf выше главной диагонали)
    #   4) softmax по последней оси устойчивым способом (вычтите максимум строки)
    #   верните (результат, матрица_внимания)
    raise NotImplementedError("Задание C.1")

def mean_entropy(d_k, scale, n=64, trials=40, seed=0):
    # TODO C.3a: средняя энтропия строк матрицы внимания для случайных Q, K
    raise NotImplementedError("Задание C.3a")

In [ ]:
# TODO C.2: проверка свойств внимания
g = np.random.default_rng(SEED)
Qt, Kt, Vt = g.normal(size=(8, 16)), g.normal(size=(8, 16)), g.normal(size=(8, 3))
_, Wt = attention(Qt, Kt, Vt, causal=True, scale=True)
# TODO C.2a: проверьте, что суммы по строкам равны 1 (assert)
# TODO C.2b: проверьте, что верхний треугольник (без диагонали) равен 0 (assert)
# TODO C.2c: напечатайте максимальное отклонение сумм строк от единицы
raise NotImplementedError("Задание C.2")

In [ ]:
# TODO C.3b: рисунок C:
#   верхний ряд — три карты внимания: двунаправленная, каузальная, без делителя (Q*4, K*4)
#   нижний ряд — энтропия внимания от d_k с делителем и без, с линией ln(64)
# используйте последовательность labels ниже: повторяющийся паттерн даёт наглядные карты
n_tok, d_k = 12, 16
labels = ["A", "B", "C", "A", "B", "C", "D", "A", "B", "E", "A", "B"]
raise NotImplementedError("Задание C.3b")

**Вопрос C.** Почему вырождение softmax в one-hot останавливает обучение? Свяжите ответ с
производной softmax по логитам.

**Ответ C.**

## Часть D. Позиционные кодировки: синусоида, RoPE, ALiBi

Внимание перестановочно инвариантно, поэтому порядок вносится отдельно:

- синусоидальная кодировка добавляется к эмбеддингу (абсолютная позиция);
- **RoPE** поворачивает $q$ и $k$ на угол, пропорциональный позиции, так что
  $\langle R_t q, R_s k\rangle = f(q,k,t-s)$;
- **ALiBi** добавляет к логитам линейный штраф $-m|t-s|$.

**Задание D.1.** Реализуйте синусоидальную кодировку.
**Задание D.2.** Реализуйте RoPE и численно проверьте зависимость только от $t-s$: постройте
$\langle R_{t}q, R_{s}k\rangle$ для нескольких абсолютных начал при одинаковых разностях позиций.
**Задание D.3.** Постройте матрицу штрафов ALiBi для одной головы и рисунок из трёх панелей.

In [ ]:
def sinusoidal(n, d):
    # TODO D.1: чётные каналы — sin(pos / 10000^(i/d)), нечётные — cos от того же аргумента
    raise NotImplementedError("Задание D.1")

def rope_apply(x, pos, base=10000.0):
    # TODO D.2: поверните пары координат (первая половина, вторая половина) на угол
    #           theta_i = pos / base ** (2i/d):
    #           x1' = x1*cos - x2*sin,  x2' = x1*sin + x2*cos
    raise NotImplementedError("Задание D.2")

# TODO D.2a: проверьте инвариантность к сдвигу абсолютных позиций и напечатайте расхождение
raise NotImplementedError("Задание D.2a")

In [ ]:
# TODO D.3: рисунок D: (1) карта синусоидальной кодировки, (2) кривые RoPE для разных абсолютных
# начал, (3) матрица штрафов ALiBi (наклоны m_h = 2^{-h}, запрещённое будущее оставьте пустым)
raise NotImplementedError("Задание D.3")

**Вопрос D.** Какая из трёх схем позволяет работать с позициями, которых не было в обучении, и
почему? Что произойдёт с обучаемой абсолютной кодировкой при выходе за максимальную длину?

**Ответ D.**

## Часть E. Мини-трансформер и индукционная голова

Задача **ассоциативного вспоминания**: последовательность пар «ключ, значение», затем повтор одного
из ключей; модель должна выдать соответствующее значение:

$
k_1\,v_1\,k_2\,v_2\,\dots\,k_N\,v_N\ \ k_i \;\longrightarrow\; v_i .
$

Минимальное решение — схема из двух слоёв (индукционная голова): первый слой переносит в позицию
ключа информацию о следующем токене, второй сравнивает финальный запрос с ключами.

**Задание E.1.** Реализуйте блок трансформера: пред-нормализация, одна голова каузального внимания,
FFN, остаточные связи.
**Задание E.2.** Обучите модель (обучение идёт на 2…`N_MAX` парах) и измерьте точность для
2…20 пар, включая длины больше обучающих.
**Задание E.3.** Постройте карты внимания обоих слоёв для одного примера и найдите индукционную
схему.

In [ ]:
N_KEYS, N_VALS = 256, 16
VOCAB = N_KEYS + N_VALS

def make_recall_batch(bs, n_pairs, gen):
    keys = torch.stack([torch.randperm(N_KEYS, generator=gen)[:n_pairs] for _ in range(bs)])
    vals = torch.randint(N_KEYS, VOCAB, (bs, n_pairs), generator=gen)
    seq = torch.zeros(bs, 2 * n_pairs + 1, dtype=torch.long)
    seq[:, 0:2 * n_pairs:2] = keys
    seq[:, 1:2 * n_pairs:2] = vals
    qi = torch.randint(0, n_pairs, (bs,), generator=gen)
    seq[:, -1] = keys[torch.arange(bs), qi]
    return seq, vals[torch.arange(bs), qi], qi

class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.q = nn.Linear(d, d, bias=False); self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False); self.o = nn.Linear(d, d, bias=False)
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, h, ret=False):
        x = self.ln1(h); T = h.shape[1]
        q, k, v = self.q(x), self.k(x), self.v(x)
        # TODO E.1: реализуйте каузальное внимание на torch:
        #   s = q @ k.transpose(1, 2) / sqrt(d)
        #   маска будущего: masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), -inf)
        #   att = s.softmax(-1);  a = att @ v
        #   затем остаточные связи: h = h + self.o(a); h = h + self.ff(self.ln2(h))
        raise NotImplementedError("Задание E.1")

class TinyTransformer(nn.Module):
    def __init__(self, d=64, max_len=90, n_layers=2):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d); self.pos = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d) for _ in range(n_layers)])
        self.lnf = nn.LayerNorm(d); self.head = nn.Linear(d, VOCAB)

    def forward(self, x, ret=False):
        h = self.emb(x) + self.pos(torch.arange(x.shape[1]))
        maps = []
        for b in self.blocks:
            h, a = b(h, ret); maps.append(a)
        return self.head(self.lnf(h)[:, -1]), maps

print("размер словаря задачи:", VOCAB, "| случайное угадывание: %.3f" % (1 / N_VALS))

In [ ]:
# TODO E.2: обучите модель и измерьте точность по числу пар
# каркас цикла обучения дан; заполните отмеченные строки
def train_recall(steps=2600, bs=64, lr=2e-3, seed=0, n_max=12):
    gen = torch.Generator().manual_seed(seed)
    m = TinyTransformer(d=D_MODEL)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.15)
    curve = []
    for s in range(steps):
        n = int(torch.randint(2, n_max + 1, (1,), generator=gen))   # смешанные длины
        x, y, _ = make_recall_batch(bs, n, gen)
        # TODO E.2a: посчитайте loss = F.cross_entropy(логиты последней позиции, y)
        # TODO E.2b: opt.zero_grad(); loss.backward(); clip_grad_norm_(..., 1.0); opt.step(); sched.step()
        raise NotImplementedError("Задание E.2")
    return m, curve, gen

def acc_vs_pairs(m, gen, ns, bs=256):
    # TODO E.2c: верните список точностей для каждого значения n из ns
    raise NotImplementedError("Задание E.2c")

NS = [2, 4, 6, 8, 10, 12, 16, 20]
t0 = time.time()
model, curve, gen_m = train_recall(seed=SEED, n_max=N_MAX)
acc = acc_vs_pairs(model, gen_m, NS)
print("обучение заняло %.0f c" % (time.time() - t0))
print("точность по числу пар:", dict(zip(NS, [round(a, 3) for a in acc])))

In [ ]:
# TODO E.3: рисунок E: кривая обучения, точность от числа пар (отметьте диапазон обучения),
# карты внимания слоёв 1 и 2 для одного примера и столбчатая диаграмма внимания финального запроса.
# Подсказка: x_ex, y_ex, qi_ex = make_recall_batch(1, 6, torch.Generator().manual_seed(123))
# и logits_ex, maps = model(x_ex, ret=True)
raise NotImplementedError("Задание E.3")

**Вопрос E.** Почему для этой задачи недостаточно одного слоя внимания? Опишите, какую информацию
переносит первый слой и что именно сравнивает второй.

**Ответ E.**

## Часть F. Линейное внимание: состояние вместо матрицы внимания

Если заменить $\exp(q^\top k)$ на $\varphi(q)^\top \varphi(k)$ с неотрицательным $\varphi$, сумму
можно переставить и получить рекуррентную форму с состоянием фиксированного размера:

$
S_t = S_{t-1} + \varphi(k_t) v_t^{\top}, \qquad
o_t = \frac{\varphi(q_t)^{\top} S_t}{\varphi(q_t)^{\top} z_t}, \qquad z_t = z_{t-1} + \varphi(k_t).
$

Генерация становится $O(1)$ по памяти, но состояние размера $d \times d$ — это фиксированная
ёмкость, в которую нужно уложить весь контекст.

**Задание F.1.** Реализуйте параллельную и рекуррентную формы линейного внимания и проверьте, что
они дают один результат.
**Задание F.2.** Измерьте ёмкость состояния: сложите в $S$ ровно $N$ пар «ключ–значение» и оцените
качество чтения (косинусная близость к истинному значению) при росте $N$.
**Задание F.3.** Постройте рисунок: рост KV-кэша softmax-внимания против постоянного состояния
линейного внимания и кривую качества чтения от числа сохранённых пар.

In [ ]:
def phi(x):
    return torch.nn.functional.elu(x) + 1          # неотрицательное отображение признаков

def linear_attention_parallel(Q, K, V):
    # TODO F.1a: параллельная форма: для каждой позиции t числитель = sum_{s<=t} phi(k_s) (phi(q_t).phi(k_s)) v_s
    #            удобная реализация: S_t = cumsum(phi(k) v^T), z_t = cumsum(phi(k)),
    #            затем o_t = phi(q_t) S_t / (phi(q_t) . z_t)
    raise NotImplementedError("Задание F.1a")

def linear_attention_recurrent(Q, K, V):
    # TODO F.1b: явный цикл по времени с состоянием S (d x d) и вектором z (d)
    raise NotImplementedError("Задание F.1b")

torch.manual_seed(SEED)
T_l, d_l = 40, 16
Ql, Kl, Vl = torch.randn(T_l, d_l), torch.randn(T_l, d_l), torch.randn(T_l, d_l)
# TODO F.1c: сравните две формы и напечатайте максимальное расхождение и размер состояния
raise NotImplementedError("Задание F.1c")

In [ ]:
# TODO F.2: ёмкость состояния
# для каждого N из NS_CAP: возьмите N случайных ключей и значений размерности d_l,
# сложите состояние S = sum phi(k_i) v_i^T, z = sum phi(k_i),
# прочитайте значение по каждому ключу и усредните косинусную близость с истинным значением
NS_CAP = [1, 2, 4, 8, 16, 32, 64, 128, 256]
raise NotImplementedError("Задание F.2")

In [ ]:
# TODO F.3: рисунок F: слева память при генерации (KV-кэш O(n) против состояния O(1)),
# справа качество чтения из состояния от числа сохранённых пар (с отметкой d_l x d_l)
raise NotImplementedError("Задание F.3")

**Вопрос F.** Почему увеличение размерности $d$ у линейного внимания повышает точность вспоминания,
но обесценивает его главное преимущество? Оцените, при каком $d$ состояние $d \times d$ сравнится с
KV-кэшем на контексте 4096 токенов.

**Ответ F.**

## Часть G. Избирательная рекуррентность

Если сделать затухание и запись зависящими от входа

$
h_t = a(x_t) \odot h_{t-1} + b(x_t) \odot u(x_t),
$

то модель получает управляемую память: она может удерживать нужное и игнорировать шум. Это ядро
идеи моделей пространства состояний с избирательностью (Mamba).

Задача: в последовательности длины 64 встречается маркер `FLAG`; нужно назвать символ,
стоящий сразу после него. Всё остальное — шум.

**Задание G.1.** Реализуйте два варианта ячейки: с гейтами, зависящими от входа, и с обучаемым, но
постоянным затуханием.
**Задание G.2.** Обучите оба варианта и сравните точность.
**Задание G.3.** Постройте профили гейтов по позициям и оцените долю каналов, у которых гейт
сохранения превышает 0.9.

In [ ]:
VOC_S, FLAG, T_S = 20, 19, 64

def flag_batch(bs, gen):
    x = torch.randint(0, 16, (bs, T_S), generator=gen)
    pos = torch.randint(2, T_S - 2, (bs,), generator=gen)
    tgt = torch.randint(0, 16, (bs,), generator=gen)
    x[torch.arange(bs), pos] = FLAG
    x[torch.arange(bs), pos + 1] = tgt
    return x, tgt, pos

class RecurCell(nn.Module):
    def __init__(self, d=48, selective=True):
        super().__init__()
        self.selective = selective
        self.emb = nn.Embedding(VOC_S, d)
        self.inp = nn.Linear(2 * d, d)                 # локальное окно ширины 2 (аналог conv1d)
        if selective:
            self.ga = nn.Linear(2 * d, d); self.gb = nn.Linear(2 * d, d)
        else:
            self.logit_a = nn.Parameter(torch.zeros(d))
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, VOC_S))

    def forward(self, x, return_gates=False):
        e = self.emb(x); B, Tn, d = e.shape
        e_prev = torch.cat([torch.zeros(B, 1, d), e[:, :-1]], 1)
        ctx = torch.cat([e, e_prev], -1)
        u = self.inp(ctx)
        # TODO G.1: получите гейты
        #   selective=True:  a = sigmoid(self.ga(ctx)), b = sigmoid(self.gb(ctx))
        #   selective=False: a = sigmoid(self.logit_a) для всех позиций, b = единицы
        # затем прогоните рекуррентность h = a[:, t] * h + b[:, t] * u[:, t]
        # и верните self.head(h), а при return_gates=True — ещё и (a, b)
        raise NotImplementedError("Задание G.1")

In [ ]:
# TODO G.2: обучите обе ячейки и сравните точность
def train_cell(selective, steps=600, bs=64, lr=5e-3, seed=0):
    gen = torch.Generator().manual_seed(seed)
    m = RecurCell(d=D_CELL, selective=selective)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    curve = []
    for s in range(steps):
        x, y, _ = flag_batch(bs, gen)
        # TODO G.2a: шаг обучения (cross_entropy, backward, clip_grad_norm_ 1.0, step)
        # TODO G.2b: каждые 25 шагов измеряйте точность на батче из 256 примеров и пишите в curve
        raise NotImplementedError("Задание G.2")
    return m, curve, gen

t0 = time.time()
cell_sel, curve_sel, gen_sel = train_cell(True, seed=SEED)
cell_fix, curve_fix, gen_fix = train_cell(False, seed=SEED)
print("избирательная ячейка: точность %.3f" % curve_sel[-1][1])
print("фиксированное затухание: точность %.3f (угадывание %.3f)" % (curve_fix[-1][1], 1 / 16))
print("обучение заняло %.0f c" % (time.time() - t0))

In [ ]:
# TODO G.3: рисунок G: кривые обучения обеих ячеек; профили гейтов (95-й перцентиль по каналам)
# с отметками позиции маркера и целевого символа; горизонт памяти log(0.01)/log(a) от a.
# Подсказка: xs, ys, pos = flag_batch(1, torch.Generator().manual_seed(77))
#            _, (a_g, b_g) = cell_sel(xs, return_gates=True)
raise NotImplementedError("Задание G.3")

**Вопрос G.** Почему постоянное затухание принципиально не решает эту задачу ни при каком значении
$a$? Рассмотрите два предельных случая: $a \to 1$ и $a \ll 1$.

**Ответ G.**

## Часть H. Декодирование, статистика попыток и мост к следующей теме

Модель задаёт распределение, а текст получается процедурой выборки. Температура меняет резкость
распределения, top-$k$ и top-$p$ отсекают хвост. Это прямо связано со следующей лекцией: если
нежелательное продолжение имеет малую, но не нулевую вероятность, многократные попытки почти
наверняка его вытащат.

**Задание H.1.** Реализуйте softmax с температурой, top-$k$ и top-$p$ отсечение.
**Задание H.2.** Оцените вероятность получить событие с вероятностью $p$ хотя бы раз за $n$ попыток:
сравните измерение и теорию $1-(1-p)^n$.
**Задание H.3.** Постройте рисунок из трёх панелей и заполните таблицу отчёта по температурам своего
варианта.
**Задание H.4.** Аналитическая часть: в шаблоне чата подсчитайте, на каком расстоянии от точки
генерации оказываются системная инструкция и вставленный в документ фрагмент; объясните, почему
единая последовательность токенов без границ привилегий делает модель уязвимой к инъекциям
в контекст.

In [ ]:
V_dec = 40
rank = np.arange(1, V_dec + 1)
logits = np.log(1.0 / rank ** 1.1)
logits = logits - logits.max()

def softmax_t(z, tau):
    # TODO H.1a: softmax с температурой, устойчиво к переполнению
    raise NotImplementedError("Задание H.1a")

def top_k_filter(p, k):
    # TODO H.1b: оставьте k самых вероятных токенов и перенормируйте
    raise NotImplementedError("Задание H.1b")

def top_p_filter(p, thr):
    # TODO H.1c: оставьте минимальный набор токенов с суммарной вероятностью >= thr
    raise NotImplementedError("Задание H.1c")

def entropy(p):
    return float(-(p * np.log(p + 1e-12)).sum())

In [ ]:
# TODO H.2: статистика многих попыток
# для p_rare в [0.001, 0.005, 0.02] и n от 20 до 360 оцените по 60 повторам частоту события
# «редкий токен выпал хотя бы раз» и сравните с теорией 1-(1-p)^n
raise NotImplementedError("Задание H.2")

In [ ]:
# TODO H.3: рисунок H: (1) распределения при температурах TAUS в логарифмической шкале,
# (2) сравнение исходного распределения с top-k и top-p, (3) вероятность события хотя бы раз
# за n попыток: теория линиями, измерение точками
raise NotImplementedError("Задание H.3")

In [ ]:
# TODO H.4: расстояния в шаблоне чата
# шаблон задан ниже как список (роль, число токенов); точка генерации — конец последовательности
TEMPLATE = [("спец", 1), ("система", 7), ("спец", 1), ("пользователь", 5),
            ("спец", 1), ("документ", 4), ("инъекция", 6), ("спец", 1)]
# TODO H.4a: посчитайте среднее расстояние от позиции каждой роли до конца последовательности
# TODO H.4b: постройте столбчатую диаграмму «роль — среднее расстояние до точки генерации»
#            и добавьте кривые относительного веса внимания exp(-m*d) для m = 0.25 и 0.03
raise NotImplementedError("Задание H.4")

**Вопрос H.** Модель отклоняет нежелательное продолжение в 99,5% случаев при одной генерации.
Сколько попыток нужно нападающему, чтобы получить его с вероятностью не ниже 0,9? Какой вывод это
даёт для методики оценки устойчивости?

**Ответ H.**

## Отчёт

### Таблица 1. Токенизация (часть A)

| Показатель | Значение |
|---|---|
| Число выполненных слияний | |
| Размер словаря токенов | |
| Сжатие «символы → BPE» | |
| Отношение «BPE → слова» | |
| Доля редких токенов (частота ≤ `RARE_THR`) | |

### Таблица 2. Затухание градиента (часть B)

| $\rho(W)$ или $f_t$ | Норма при $k=100$ | Горизонт до $10^{-6}$ |
|---|---|---|
| 0.9 | | |
| 1.0 | | |
| 1.2 | | |
| $f_t = 0.99$ | | |

### Таблица 3. Масштабирование внимания (часть C)

| $d_k$ | Энтропия с делителем | Энтропия без делителя |
|---|---|---|
| 8 | | |
| 64 | | |
| 256 | | |

### Таблица 4. Мини-трансформер (часть E)

| Число пар в контексте | 2 | 4 | 8 | 12 | 16 | 20 |
|---|---|---|---|---|---|---|
| Точность вспоминания | | | | | | |

Финальное значение функции потерь: ______  Время обучения: ______ с
Масса внимания финального запроса на позиции нужного значения: ______

### Таблица 5. Линейное внимание (часть F)

| Показатель | Значение |
|---|---|
| Расхождение параллельной и рекуррентной форм | |
| Размер состояния, чисел | |
| Качество чтения при 4 парах | |
| Качество чтения при 256 парах | |

### Таблица 6. Избирательная рекуррентность (часть G)

| Вариант ячейки | Точность | Комментарий |
|---|---|---|
| избирательные гейты | | |
| постоянное затухание | | |
| случайное угадывание | 0.062 | теоретическое значение |

Доля каналов с гейтом сохранения выше 0.9: ______

### Таблица 7. Декодирование (часть H)

| Температура варианта | Энтропия, нат | $p_{\max}$ |
|---|---|---|
| | | |
| | | |
| | | |

Число попыток для события с $p=0{,}005$ до вероятности 0,9: ______

### Выводы

Сформулируйте не менее пяти выводов. Каждый вывод должен опираться на конкретное измеренное число
из вашего прогона.

1.
2.
3.
4.
5.

## Контрольные вопросы

1. Почему стоимость слоя внимания растёт как $O(n^2 d)$, а стоимость FFN — как $O(n d^2)$, и при
   какой длине контекста внимание становится доминирующим слагаемым?
2. Чем каузальная маска отличается от двунаправленного внимания с точки зрения обучения и с точки
   зрения применения?
3. Почему RoPE позволяет расширять контекст после обучения, а обучаемая абсолютная кодировка — нет?
4. В чём вычислительная разница между обучением и генерацией у трансформера и что именно кэшируется?
5. Как GQA и MQA уменьшают KV-кэш и чем за это приходится платить?
6. Почему линейное внимание проигрывает softmax-вниманию на задачах точного вспоминания?
7. Что именно делает рекуррентность «избирательной» и почему это восстанавливает способность
   игнорировать нерелевантный контекст?
8. Почему смесь экспертов требует балансирующей потери и что происходит без неё?
9. Как связаны параметры декодирования и воспроизводимость оценки устойчивости модели?
10. Почему шаблон чата не создаёт настоящей границы привилегий между системной инструкцией и
    содержимым документа?

## Дополнительные задания

- Добавьте в часть E третий слой и проверьте, меняется ли характер переноса на длины больше
  обучающих.
- Замените в части E обучаемую абсолютную кодировку на RoPE и повторите измерение точности для
  16 и 20 пар.
- Реализуйте в части F многоголовое линейное внимание и постройте зависимость качества чтения от
  числа голов при фиксированном общем размере состояния.
- Реализуйте слой смеси экспертов с балансирующей потерей и покажите, как меняется распределение
  нагрузки при её отключении.
- Постройте в части H оценку числа попыток до заданной вероятности успеха как функцию $p$ и
  обсудите, как это влияет на протокол тестирования.

## Чек-лист сдачи

- [ ] Заполнен паспорт работы, указан вариант и его параметры.
- [ ] Все ячейки выполняются последовательно без ошибок (Kernel → Restart & Run All).
- [ ] Реализованы все `TODO` в частях A–H.
- [ ] Заполнены таблицы 1–7 числами своего варианта.
- [ ] Даны ответы на вопросы A–H и контрольные вопросы 1–10.
- [ ] Сформулировано не менее пяти выводов со ссылками на измеренные значения.
- [ ] Все графики подписаны по-русски и читаемы.



## Литература

- Vaswani A. et al. Attention Is All You Need (2017): <https://arxiv.org/abs/1706.03762>
- Hochreiter S., Schmidhuber J. Long Short-Term Memory (1997):
  <https://direct.mit.edu/neco/article/9/8/1735/6109>
- Su J. et al. RoFormer: Enhanced Transformer with Rotary Position Embedding (2021):
  <https://arxiv.org/abs/2104.09864>
- Press O. et al. Train Short, Test Long: Attention with Linear Biases (ALiBi) (2021):
  <https://arxiv.org/abs/2108.12409>
- Katharopoulos A. et al. Transformers are RNNs: Fast Autoregressive Transformers with Linear
  Attention (2020): <https://arxiv.org/abs/2006.16236>
- Gu A., Dao T. Mamba: Linear-Time Sequence Modeling with Selective State Spaces (2023):
  <https://arxiv.org/abs/2312.00752>
- Ainslie J. et al. GQA: Training Generalized Multi-Query Transformer Checkpoints (2023):
  <https://arxiv.org/abs/2305.13245>
- Dao T. et al. FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness (2022):
  <https://arxiv.org/abs/2205.14135>
- Olsson C. et al. In-context Learning and Induction Heads (2022):
  <https://arxiv.org/abs/2209.11895>
- Holtzman A. et al. The Curious Case of Neural Text Degeneration (2019):
  <https://arxiv.org/abs/1904.09751>